In [1]:
import pandas as pd

features_df = pd.read_csv(
    r"D:\UPI-Fraud-Detection\data\processed\features.csv"
)

alerts_df = pd.read_csv(
    r"D:\UPI-Fraud-Detection\data\processed\alerts.csv"
)

print("Features:", features_df.shape)
print("Alerts:", alerts_df.shape)

Features: (10000, 50)
Alerts: (399, 50)


In [2]:
from fastapi.encoders import jsonable_encoder

transaction_id = str(features_df.iloc[0]["transaction_id"])

result = features_df[
    features_df["transaction_id"].astype(str) == transaction_id
]

record = result.iloc[0].to_dict()

print("Transaction ID:", transaction_id)
print("Rows found:", len(result))

encoded_record = jsonable_encoder(record)

print("Transaction JSON encoding: SUCCESS")

Transaction ID: TXN_000898
Rows found: 1
Transaction JSON encoding: SUCCESS


In [3]:
alert_records = alerts_df.to_dict(orient="records")

encoded_alerts = jsonable_encoder(alert_records)

print("Alerts JSON encoding: SUCCESS")
print("Total alerts:", len(encoded_alerts))

Alerts JSON encoding: SUCCESS
Total alerts: 399


In [4]:
import api.main

print("FastAPI is loading main.py from:")
print(api.main.__file__)

FastAPI is loading main.py from:
D:\UPI-Fraud-Detection\api\main.py


In [5]:
print("Transaction JSON encoding: SUCCESS")
print("Transaction ID:", transaction_id)

Transaction JSON encoding: SUCCESS
Transaction ID: TXN_000898


In [6]:
print("Alerts JSON encoding: SUCCESS")
print("Total alerts:", len(encoded_alerts))

Alerts JSON encoding: SUCCESS
Total alerts: 399


In [7]:
import api.main

print("Loaded main.py:")
print(api.main.__file__)

Loaded main.py:
D:\UPI-Fraud-Detection\api\main.py


In [8]:
import inspect
import api.main

print(inspect.getsource(api.main.get_alerts))
print("\n" + "=" * 60 + "\n")
print(inspect.getsource(api.main.get_transaction))

@app.get("/alerts")
def get_alerts():
    records = alerts_df.to_dict(
        orient="records"
    )

    return {
        "total_alerts": len(records),
        "alerts": jsonable_encoder(records)
    }



@app.get("/transaction/{transaction_id}")
def get_transaction(
    transaction_id: str
):
    result = features_df[
        features_df["transaction_id"].astype(str)
        == str(transaction_id)
    ]

    if result.empty:
        raise HTTPException(
            status_code=404,
            detail="Transaction not found"
        )

    record = result.iloc[0].to_dict()

    return jsonable_encoder(record)



In [9]:
import requests

transaction_id = str(features_df.iloc[0]["transaction_id"])

response1 = requests.get(
    f"http://127.0.0.1:8000/transaction/{transaction_id}"
)

print("Transaction endpoint:")
print("Status:", response1.status_code)
print(response1.text[:1000])

print("\n" + "=" * 60 + "\n")

response2 = requests.get(
    "http://127.0.0.1:8000/alerts"
)

print("Alerts endpoint:")
print("Status:", response2.status_code)
print(response2.text[:1000])

Transaction endpoint:
Status: 500
Internal Server Error


Alerts endpoint:
Status: 500
Internal Server Error


In [10]:
import requests

response = requests.get(
    "http://127.0.0.1:8000/alerts"
)

print("Status:", response.status_code)
print("Response:", response.text)

Status: 500
Response: Internal Server Error


In [11]:
from pathlib import Path

main_path = Path(
    r"D:\UPI-Fraud-Detection\api\main.py"
)

main_code = main_path.read_text(encoding="utf-8")

# Add numpy import if not already present
if "import numpy as np" not in main_code:
    main_code = main_code.replace(
        "from fastapi import",
        "import numpy as np\n\nfrom fastapi import"
    )

# Add helper function before the first route
helper = '''

def clean_for_json(data):
    if isinstance(data, list):
        return [clean_for_json(item) for item in data]

    if isinstance(data, dict):
        return {
            key: clean_for_json(value)
            for key, value in data.items()
        }

    if isinstance(data, float):
        if not np.isfinite(data):
            return None
        return data

    return data
'''

if "def clean_for_json(data):" not in main_code:
    marker = "@app.get("
    main_code = main_code.replace(
        marker,
        helper + "\n\n" + marker,
        1
    )

# Replace the two return statements
main_code = main_code.replace(
    'return jsonable_encoder(records)',
    'return clean_for_json(jsonable_encoder(records))'
)

main_code = main_code.replace(
    'return jsonable_encoder(record)',
    'return clean_for_json(jsonable_encoder(record))'
)

main_path.write_text(main_code, encoding="utf-8")

print("✓ main.py updated successfully")
print(main_path)

✓ main.py updated successfully
D:\UPI-Fraud-Detection\api\main.py


In [12]:
with open(
    r"D:\UPI-Fraud-Detection\api\main.py",
    "r",
    encoding="utf-8"
) as file:
    print(file.read())


import numpy as np

from fastapi import FastAPI, HTTPException
from fastapi.encoders import jsonable_encoder

from .schemas import (
    TransactionRequest,
    PredictionResponse
)

from .predictor import (
    calculate_prediction,
    alerts_df,
    features_df
)


app = FastAPI(
    title="UPI Fraud Detection API",
    description="API for UPI transaction anomaly and fraud risk detection",
    version="1.0.0"
)




def clean_for_json(data):
    if isinstance(data, list):
        return [clean_for_json(item) for item in data]

    if isinstance(data, dict):
        return {
            key: clean_for_json(value)
            for key, value in data.items()
        }

    if isinstance(data, float):
        if not np.isfinite(data):
            return None
        return data

    return data


@app.get("/health")
def health_check():
    return {
        "status": "healthy",
        "service": "UPI Fraud Detection API"
    }


@app.post(
    "/predict",
    response_model=PredictionRe

In [13]:
import pandas as pd
import numpy as np

features_df = pd.read_csv(
    r"D:\UPI-Fraud-Detection\data\processed\features.csv"
)

alerts_df = pd.read_csv(
    r"D:\UPI-Fraud-Detection\data\processed\alerts.csv"
)

print("Features NaN values:", features_df.isna().sum().sum())
print("Features infinite values:", np.isinf(
    features_df.select_dtypes(include=np.number)
).sum().sum())

print("Alerts NaN values:", alerts_df.isna().sum().sum())
print("Alerts infinite values:", np.isinf(
    alerts_df.select_dtypes(include=np.number)
).sum().sum())

Features NaN values: 2000
Features infinite values: 0
Alerts NaN values: 118
Alerts infinite values: 0


In [14]:
import requests

transaction_id = str(features_df.iloc[0]["transaction_id"])

response = requests.get(
    f"http://127.0.0.1:8000/transaction/{transaction_id}"
)

print("Status Code:", response.status_code)
print(response.text[:3000])

Status Code: 200
{"transaction_id":"TXN_000898","timestamp":"2026-09-14 04:51:42.793653","sender_id":"USER_0001","receiver_id":"USER_0451","amount":120.22,"merchant_category":"Education","transaction_type":"P2M","location":"Chennai","device_type":"Android","upi_channel":"PhonePe","hour":4,"day_of_week":0,"is_weekend":0,"is_late_night":1,"amount_deviation":0.2289389649233684,"transactions_last_1h":0.0,"recipient_repeat_count":0,"previous_device":null,"previous_location":null,"device_changed":0,"location_changed":0,"fraud_label":0,"user_avg_amount":525.118125,"user_median_amount":299.97,"user_max_amount":2385.88,"user_transaction_count":16,"iqr_anomaly":0,"isolation_prediction":1,"isolation_anomaly":0,"isolation_score":-0.0668463935730294,"transaction_hour":"2026-09-14 04:00:00","time_series_anomaly":0,"burst_anomaly":0,"risk_iqr":0,"risk_isolation":0,"risk_burst":0,"risk_device":0,"risk_location":0,"risk_late_night":5,"risk_time_series":0,"risk_score":5,"risk_level":"Low","is_suspicious

In [15]:
response = requests.get(
    "http://127.0.0.1:8000/alerts"
)

print("Status Code:", response.status_code)
print(response.text[:3000])

Status Code: 500
Internal Server Error


In [16]:
from pathlib import Path

main_path = Path(
    r"D:\UPI-Fraud-Detection\api\main.py"
)

main_code = main_path.read_text(encoding="utf-8")

start = main_code.find('@app.get("/alerts")')
end = main_code.find('@app.get("/transaction/{transaction_id}")')

if start == -1 or end == -1:
    print("❌ Could not locate the /alerts or /transaction endpoint.")
else:
    new_alerts_endpoint = '''@app.get("/alerts")
def get_alerts():
    records = alerts_df.to_dict(orient="records")

    clean_records = []

    for record in records:
        clean_record = {}

        for key, value in record.items():

            # Handle missing values
            if value is None:
                clean_record[key] = None
                continue

            try:
                if pd.isna(value):
                    clean_record[key] = None
                    continue
            except (TypeError, ValueError):
                pass

            # Handle NumPy integer values
            if isinstance(value, np.integer):
                clean_record[key] = int(value)
                continue

            # Handle NumPy and Python floating-point values
            if isinstance(value, (np.floating, float)):
                if np.isfinite(value):
                    clean_record[key] = float(value)
                else:
                    clean_record[key] = None
                continue

            # Handle normal integers
            if isinstance(value, int):
                clean_record[key] = value
                continue

            # Handle normal strings and other JSON-safe values
            clean_record[key] = value

        clean_records.append(clean_record)

    return {
        "total_alerts": len(clean_records),
        "alerts": clean_records
    }


'''

    main_code = (
        main_code[:start]
        + new_alerts_endpoint
        + main_code[end:]
    )

    main_path.write_text(main_code, encoding="utf-8")

    print("✅ /alerts endpoint completely replaced.")
    print("File updated:")
    print(main_path)

✅ /alerts endpoint completely replaced.
File updated:
D:\UPI-Fraud-Detection\api\main.py


In [17]:
import inspect
import api.main

print(inspect.getsource(api.main.get_alerts))

def clean_for_json(data):
    if isinstance(data, list):
        return [clean_for_json(item) for item in data]

    if isinstance(data, dict):
        return {
            key: clean_for_json(value)
            for key, value in data.items()
        }

    if isinstance(data, float):
        if not np.isfinite(data):
            return None
        return data

    return data



In [18]:
import requests

response = requests.get(
    "http://127.0.0.1:8000/alerts"
)

print("Status Code:", response.status_code)
print(response.text[:3000])

Status Code: 500
Internal Server Error


In [19]:
import requests

response = requests.get(
    "http://127.0.0.1:8000/alerts"
)

print("Status Code:", response.status_code)
print(response.text[:2000])

Status Code: 200
{"total_alerts":399,"alerts":[{"transaction_id":"TXN_007856","timestamp":"2026-09-19 00:49:42.793653","sender_id":"USER_0855","receiver_id":"USER_0436","amount":1764.06,"merchant_category":"Travel","transaction_type":"P2P","location":"Chennai","device_type":"Web","upi_channel":"Other","hour":0,"day_of_week":5,"is_weekend":1,"is_late_night":1,"amount_deviation":3.5291678913892897,"transactions_last_1h":18.0,"recipient_repeat_count":0,"previous_device":"iOS","previous_location":"Hyderabad","device_changed":1,"location_changed":1,"fraud_label":1,"user_avg_amount":499.8515384615384,"user_median_amount":403.15,"user_max_amount":1764.06,"user_transaction_count":13,"iqr_anomaly":1,"isolation_prediction":-1,"isolation_anomaly":1,"isolation_score":0.1386047732723605,"transaction_hour":"2026-09-19 00:00:00","time_series_anomaly":0,"burst_anomaly":1,"risk_iqr":20,"risk_isolation":30,"risk_burst":20,"risk_device":15,"risk_location":10,"risk_late_night":5,"risk_time_series":0,"risk

In [20]:
import requests

response = requests.get(
    "http://127.0.0.1:8000/alerts"
)

print("Status Code:", response.status_code)
print(response.text[:2000])

Status Code: 200
{"total_alerts":399,"alerts":[{"transaction_id":"TXN_007856","timestamp":"2026-09-19 00:49:42.793653","sender_id":"USER_0855","receiver_id":"USER_0436","amount":1764.06,"merchant_category":"Travel","transaction_type":"P2P","location":"Chennai","device_type":"Web","upi_channel":"Other","hour":0,"day_of_week":5,"is_weekend":1,"is_late_night":1,"amount_deviation":3.5291678913892897,"transactions_last_1h":18.0,"recipient_repeat_count":0,"previous_device":"iOS","previous_location":"Hyderabad","device_changed":1,"location_changed":1,"fraud_label":1,"user_avg_amount":499.8515384615384,"user_median_amount":403.15,"user_max_amount":1764.06,"user_transaction_count":13,"iqr_anomaly":1,"isolation_prediction":-1,"isolation_anomaly":1,"isolation_score":0.1386047732723605,"transaction_hour":"2026-09-19 00:00:00","time_series_anomaly":0,"burst_anomaly":1,"risk_iqr":20,"risk_isolation":30,"risk_burst":20,"risk_device":15,"risk_location":10,"risk_late_night":5,"risk_time_series":0,"risk

In [21]:
import requests

base_url = "http://127.0.0.1:8000"

# 1. Health
r1 = requests.get(f"{base_url}/health")
print("Health:", r1.status_code, r1.json())

# 2. Alerts
r2 = requests.get(f"{base_url}/alerts")
print("Alerts:", r2.status_code)

# 3. Transaction
transaction_id = str(features_df.iloc[0]["transaction_id"])
r3 = requests.get(f"{base_url}/transaction/{transaction_id}")
print("Transaction:", r3.status_code)

# 4. Root
r4 = requests.get(f"{base_url}/")
print("Root:", r4.status_code, r4.json())

Health: 200 {'status': 'healthy', 'service': 'UPI Fraud Detection API'}
Alerts: 200
Transaction: 200
Root: 200 {'message': 'UPI Fraud Detection API', 'docs': '/docs', 'health': '/health'}


In [1]:
import requests

url = "http://127.0.0.1:8000/predict"

transaction = {
    "amount": 25000,
    "transactions_last_1h": 8,
    "recipient_repeat_count": 0,
    "device_changed": 1,
    "location_changed": 1,
    "hour": 2,
    "day_of_week": 5,
    "is_weekend": 1,
    "is_late_night": 1
}

response = requests.post(
    url,
    json=transaction
)

print("Status Code:", response.status_code)
print(response.json())

Status Code: 422
{'detail': [{'type': 'missing', 'loc': ['body', 'transaction_id'], 'msg': 'Field required', 'input': {'amount': 25000, 'transactions_last_1h': 8, 'recipient_repeat_count': 0, 'device_changed': 1, 'location_changed': 1, 'hour': 2, 'day_of_week': 5, 'is_weekend': 1, 'is_late_night': 1}}, {'type': 'missing', 'loc': ['body', 'amount_deviation'], 'msg': 'Field required', 'input': {'amount': 25000, 'transactions_last_1h': 8, 'recipient_repeat_count': 0, 'device_changed': 1, 'location_changed': 1, 'hour': 2, 'day_of_week': 5, 'is_weekend': 1, 'is_late_night': 1}}]}


In [2]:
import requests

url = "http://127.0.0.1:8000/predict"

transaction = {
    "transaction_id": "TEST_000001",
    "amount": 25000,
    "amount_deviation": 4.5,
    "transactions_last_1h": 8,
    "recipient_repeat_count": 0,
    "device_changed": 1,
    "location_changed": 1,
    "hour": 2,
    "day_of_week": 5,
    "is_weekend": 1,
    "is_late_night": 1
}

response = requests.post(
    url,
    json=transaction
)

print("Status Code:", response.status_code)
print(response.json())

Status Code: 200
{'transaction_id': 'TEST_000001', 'risk_score': 90.0, 'risk_level': 'Critical', 'is_suspicious': True, 'reasons': ['Unusually high or low transaction amount', 'Isolation Forest detected anomalous behavior', 'High transaction frequency within 1 hour', 'New or changed device detected', 'Location change detected', 'Late-night transaction'], 'isolation_anomaly': 1, 'iqr_anomaly': 1, 'burst_anomaly': 1}
